In [265]:
# importons les bibliothèques 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import  accuracy_score, classification_report

from imblearn.over_sampling import SMOTE




In [226]:
# Chargeons nos données
df = pd.read_csv("../data/raw/healthcare-dataset-stroke-data.csv")

print(df.head())

      id  gender   age  ...   bmi   smoking_status stroke
0   9046    Male  67.0  ...  36.6  formerly smoked      1
1  51676  Female  61.0  ...   NaN     never smoked      1
2  31112    Male  80.0  ...  32.5     never smoked      1
3  60182  Female  49.0  ...  34.4           smokes      1
4   1665  Female  79.0  ...  24.0     never smoked      1

[5 rows x 12 columns]


In [227]:
# nous allons faire une copie de notre dataset pour éviter de l'endommager
df1 = df.copy()

In [228]:
# nous allons supprimer la colonne "id" qui n'est pas un paramètre médical ici, mais d'identification des patients de manière anonyme
df1 = df1.drop("id", axis=1)

print(df1.head())

   gender   age  hypertension  ...   bmi   smoking_status stroke
0    Male  67.0             0  ...  36.6  formerly smoked      1
1  Female  61.0             0  ...   NaN     never smoked      1
2    Male  80.0             0  ...  32.5     never smoked      1
3  Female  49.0             0  ...  34.4           smokes      1
4  Female  79.0             1  ...  24.0     never smoked      1

[5 rows x 11 columns]


#### Création des variables X et y 

In [229]:
# Défissons les variables X et y
X=df1.drop(columns="stroke") # variables explicatives
y= df1["stroke"]                # variable cible


Les caractéristiques (X) du patient nous permettrons de prédire la présence ou non d'un AVC (y) à l'aide d'un algorithme (Modèle du Machine Learning).

In [230]:
# vérifions
print(f" X se représente ainsi: \n {X.head()}")

print()

print(f" y se présente ainsi: \n {y.head()}")

 X se représente ainsi: 
    gender   age  hypertension  ...  avg_glucose_level   bmi   smoking_status
0    Male  67.0             0  ...             228.69  36.6  formerly smoked
1  Female  61.0             0  ...             202.21   NaN     never smoked
2    Male  80.0             0  ...             105.92  32.5     never smoked
3  Female  49.0             0  ...             171.23  34.4           smokes
4  Female  79.0             1  ...             174.12  24.0     never smoked

[5 rows x 10 columns]

 y se présente ainsi: 
 0    1
1    1
2    1
3    1
4    1
Name: stroke, dtype: int64


In [231]:
print(f"X est composé de: {X.shape[0]} lignes et {X.shape[1]} colonnes")
print(f"y est composé de: {y.shape[0]} lignes")

X est composé de: 5110 lignes et 10 colonnes
y est composé de: 5110 lignes


y = 5110 est une série, un seul vecteur de 5 110 valeurs, ce qui est exactement ce qu'attendent les modèles de scikit-learn en tant que variable cible.



#### Séparation du jeu d'entrainement et du jeu de test (jeu de test à 20% et random_state = 42)
Nous allons ajouter stratify = y car notre dataset contient un jeu de donnée déséquilibré :
95 % de non AVC et 5 % d'AVC

Comme nous allons séparer les données au hasard, on pourrait obtenir un jeu de test avec très peu de patients ayant subi un AVC. Le modèle serait alors mal évalué.

d'où pour conserver la même proportion d'AVC dans les deux jeux, on ajoutera stratify=y.

In [232]:
X_train, X_test, y_train, y_test = train_test_split(
                                        X, 
                                        y, 
                                        test_size = 0.2, 
                                        random_state = 42,
                                        stratify = y
                                        )


In [233]:
# vérifions
print("X_train: ", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train: ", y_train.shape)
print("y_test: ", y_test.shape)

X_train:  (4088, 10)
X_test:  (1022, 10)
y_train:  (4088,)
y_test:  (1022,)


In [234]:
# vérifions que la proportion des classes est bien concervée
print((y_train.value_counts(normalize=True)*100).round())

stroke
0    95.0
1     5.0
Name: proportion, dtype: float64


In [235]:
print((y_test.value_counts(normalize=True)*100).round())

stroke
0    95.0
1     5.0
Name: proportion, dtype: float64


#### Feature Engineering: Identifier les types de variables manuellement

In [236]:
# variables numériques

num_features = ["age", 
                "avg_glucose_level", 
                "bmi"]

# variables catégorielles

cat_features = ["gender", "ever_married", 
                "work_type", "Residence_type", 
                "smoking_status" ]

# variables numériques binaies

binary_features = ["hypertension", 
                    "heart_disease"]
                    



#### Nous allons nous intéresser à nos valeurs manquantes de la variable `bmi`
Etant donné que notre feature "bmi" a une distribution asymétrique, nous allons imputer nos valeurs manquantes avec la médiane qui est plus robuste que la moyenne.

Pour éviter le data leakage, nous allons utiliser la médiane de notre jeu d'entrainemeent puis l'appliquer directement à notre jeu de test 

In [237]:
# médiane jeu d'entrainement
median_bmi = X_train["bmi"].median()

print(f"la mediane du X_train est de: {median_bmi}")

la mediane du X_train est de: 28.0


In [238]:
# imputation médiane au train et test
X_train["bmi"] = X_train["bmi"].fillna(median_bmi)

X_test["bmi"] = X_test["bmi"].fillna(median_bmi)


In [239]:
# vérifions
print(X_train.isna().sum())

gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
dtype: int64


In [240]:
print(X_test.isna().sum())

gender               0
age                  0
hypertension         0
heart_disease        0
ever_married         0
work_type            0
Residence_type       0
avg_glucose_level    0
bmi                  0
smoking_status       0
dtype: int64


#### Encodage des variables catégorielles

In [241]:
# Créons l'encodeur

encoder = OneHotEncoder(drop = "first",
                        handle_unknown = "ignore",
                        sparse_output = False
                        )

In [242]:
# nous allons encoder notre jeu d'entrainement
X_train_encoded = encoder.fit_transform(
                                X_train[cat_features]
                                )

In [243]:
# appliquons ce qui a été appris au jeu de test
X_test_encoded = encoder.transform(
                X_test[cat_features]
                    )

In [244]:
print(X_train_encoded)

[[0. 0. 1. ... 0. 1. 0.]
 [1. 0. 0. ... 0. 1. 0.]
 [0. 0. 1. ... 0. 1. 0.]
 ...
 [0. 0. 1. ... 1. 0. 0.]
 [1. 0. 1. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [245]:
print(X_test_encoded)

[[1. 0. 1. ... 0. 1. 0.]
 [0. 0. 1. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 0. 1.]
 ...
 [0. 0. 1. ... 0. 1. 0.]
 [1. 0. 0. ... 0. 1. 0.]
 [0. 0. 1. ... 1. 0. 0.]]


In [246]:
# Conversion des tableaux NumPy en DataFrame

encoded_columns = encoder.get_feature_names_out(cat_features)

X_train_encoded = pd.DataFrame(
                    X_train_encoded,
                    columns = encoded_columns,
                    index = X_train.index
                    )

X_train_encoded.head()

,gender_Male,gender_Other,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
845,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3744,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4183,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3409,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
284,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [247]:
# vérifions que tout s'est bien déroulé
print(X_train_encoded.isna().sum())

gender_Male                       0
gender_Other                      0
ever_married_Yes                  0
work_type_Never_worked            0
work_type_Private                 0
work_type_Self-employed           0
work_type_children                0
Residence_type_Urban              0
smoking_status_formerly smoked    0
smoking_status_never smoked       0
smoking_status_smokes             0
dtype: int64


In [248]:
X_test_encoded = pd.DataFrame(X_test_encoded,
                                columns = encoded_columns,
                                index = X_test.index
                                )
X_test_encoded.head()

,gender_Male,gender_Other,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
3725,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4481,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1545,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
1820,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1262,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [249]:
# vérifions
X_test_encoded.isna().sum()

gender_Male                       0
gender_Other                      0
ever_married_Yes                  0
work_type_Never_worked            0
work_type_Private                 0
work_type_Self-employed           0
work_type_children                0
Residence_type_Urban              0
smoking_status_formerly smoked    0
smoking_status_never smoked       0
smoking_status_smokes             0
dtype: int64

In [250]:
# Nous allons récupérer nos variables numériques préalablement identifiées
# jeu d'entrainement
X_train_num = X_train[num_features]
# jeu de test

X_test_num = X_test[num_features]

In [251]:
# vérifions
print(X_train_num.head())

       age  avg_glucose_level   bmi
845   48.0              69.21  33.1
3744  15.0             122.25  21.0
4183  67.0             110.42  24.9
3409  44.0              65.41  24.8
284   14.0              82.34  31.6


In [252]:
# Nous allons récupérer nos variables binaires préalablement identifiées
# jeu d'entrainement
X_train_bin = X_train[binary_features]
# jeu de test
X_test_bin = X_test[binary_features]

In [253]:
# vérifions
print(X_train_bin.head())

      hypertension  heart_disease
845              0              0
3744             0              0
4183             0              0
3409             0              0
284              0              0


In [254]:
# Nous allons fusionner nos différentes tables du jeu d'entrainement par concaténation
X_train_final = pd.concat(
                [
                    X_train_num,
                    X_train_bin,
                    X_train_encoded
                ], axis=1
                )
X_train_final.head()

,age,avg_glucose_level,bmi,hypertension,heart_disease,gender_Male,gender_Other,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
845,48.0,69.21,33.1,0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3744,15.0,122.25,21.0,0,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4183,67.0,110.42,24.9,0,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3409,44.0,65.41,24.8,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
284,14.0,82.34,31.6,0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [255]:
# vérifions
print(X_train_final.shape)


(4088, 16)


In [256]:
# Nous allons fusionner nos différentes tables du jeu de test par concaténation
X_test_final = pd.concat(
                [
                    X_test_num,
                    X_test_bin,
                    X_test_encoded
                ], axis=1
                )
X_test_final.head()

,age,avg_glucose_level,bmi,hypertension,heart_disease,gender_Male,gender_Other,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
3725,63.0,78.23,34.8,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4481,43.0,86.67,33.3,0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1545,23.0,126.67,28.7,0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
1820,21.0,208.17,24.9,0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1262,67.0,113.34,26.3,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [257]:
# vérifions
print(X_test_final.shape)

(1022, 16)


Afin de préparer notre dataset final qu'il soit compatible avec tout type de model du machine learning que nous déciderons d'utiliser, nous allons standardiser nos valeurs numériques.

In [258]:
#le model utilisé pour Standardiser 
scaler = StandardScaler()


In [259]:
# appliquons notre model en transformant nos variables numériques 
X_train_final[num_features] = scaler.fit_transform(
                                X_train_final[num_features]
                                )

In [260]:
# appliquons sur le test
X_test_final[num_features] = scaler.transform(
                            X_test_final[num_features]
                                )

In [261]:
# Vérification
X_train_final.head()

,age,avg_glucose_level,bmi,hypertension,heart_disease,gender_Male,gender_Other,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
845,0.205661,-0.819973,0.543113,0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
3744,-1.254901,0.352075,-1.015569,0,0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4183,1.046590,0.090662,-0.513184,0,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3409,0.028623,-0.903944,-0.526066,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
284,-1.299160,-0.529834,0.349888,0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [262]:
X_test_final.head()

,age,avg_glucose_level,bmi,hypertension,heart_disease,gender_Male,gender_Other,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
3725,0.869552,-0.620654,0.762101,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4481,-0.015636,-0.434152,0.568876,0,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1545,-0.900825,0.449745,-0.023680,0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
1820,-0.989344,2.250687,-0.513184,0,0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
1262,1.046590,0.155187,-0.332841,0,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [263]:
X_train_final.describe().round(2)

,age,avg_glucose_level,bmi,hypertension,heart_disease,gender_Male,gender_Other,ever_married_Yes,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Urban,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
count,4088.00,4088.00,4088.00,4088.0,4088.00,4088.00,4088.00,4088.00,4088.00,4088.00,4088.00,4088.00,4088.00,4088.00,4088.00,4088.00
mean,-0.00,0.00,-0.00,0.1,0.05,0.41,0.00,0.66,0.00,0.57,0.16,0.14,0.51,0.17,0.37,0.15
std,1.00,1.00,1.00,0.3,0.23,0.49,0.02,0.47,0.06,0.50,0.37,0.34,0.50,0.38,0.48,0.36
min,-1.92,-1.13,-2.39,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,-0.77,-0.64,-0.65,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,0.07,-0.32,-0.11,0.0,0.00,0.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00
75%,0.78,0.17,0.50,0.0,0.00,1.00,0.00,1.00,0.00,1.00,0.00,0.00,1.00,0.00,1.00,0.00
max,1.71,3.66,8.85,1.0,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.00


La standardisation s'est bien déroulée car on observe que nos trois variables numériques ont une moyenne proche de 0 et un écart-type proche de 1.

#### Gérer le déséquilibre des classes avec SMOTE

In [268]:
# déséquilibre de notre variable cible
y_train.value_counts(normalize=True).round(2)*100

stroke
0    95.0
1     5.0
Name: proportion, dtype: float64

Nous comprendrons que sur 100 patients: 95 n'ont pas fait d'AVC et seulement 5 ont fait un AVC.
Alors on aura un gros problème lors de la prédiction, car certains modèles pourraient répondre à chaque fois pas AVC ce qui ne serait pas correct.
Nous avons des solutions:
- réduire les patients sans AVC mais en contrepartis, sous-echantillange: on perdra des données,
- copier et coller nos patients ayant fait un AVC problème, nous aurons des doublons et conséquence: surapprentissage par les modèles
- d'où la solution qui nous convient le mieux, SMOTE (Synthetic Minority Over-sampling Technique): qui nous permettra de fabriquer de nouveaux patients artificiels au lieu de copier les patients AVC. 
Ce qui nous donnera de nouveaux exemples plausibles. Le modèle apprendra donc beaucoup mieux.

SMOTE ne sera appliqué qu'au jeu d'entrainement. Ainsi nous éviterons de biaiser le jeu de test.


In [ ]:
# importons SMOTE
smote = SMOTE(random_state=42) # Nous avons garder le random_state à 42 comme depuis le départ afin d'avoir le même échantillonnage.


In [ ]:
# appliquons la technique sur notre jeu d'entrainement
X_train_smote, y_train_smote = smote.fit_resample(
                                X_train_final, 
                                y_train
                                ) # SMOTE apprend la distribution puis crée les nouvelles observations.

In [273]:
# Vérifications: nous pouvons observer la différence entre la variable cible sans smote et celle avec
print("Avant SMOTE")
print(y_train.value_counts())

print("\nAprès SMOTE")
print(y_train_smote.value_counts())

Avant SMOTE
stroke
0    3889
1     199
Name: count, dtype: int64

Après SMOTE
stroke
0    3889
1    3889
Name: count, dtype: int64


### Nous aurons enfin un jeu de données prêt à entraîner nos modèles.